# Clase 142 — Mecanismos de atención

La **atención** destrabó el NLP moderno: en vez de un único vector de contexto, el
modelo pondera **todos** los estados relevantes. El corazón es la **scaled
dot-product attention**: `softmax(QK^T / √d) · V`.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`. Se ejecuta en Colab con GPU.

## 🧠 Intuición previa: atención = cada token decide a quién mirar

Imaginá que estás traduciendo la frase *"el gato negro"* y tenés que producir la palabra *"black"*. No mirás toda la oración por igual: **prestás atención** sobre todo a *"negro"*. Eso es la atención: para cada posición, el modelo **decide a cuáles otras posiciones mirar y con qué peso**.

El mecanismo es una búsqueda blanda (*soft lookup*): cada token emite una **query** (qué busco), cada token ofrece una **key** (qué tengo) y un **value** (qué aporto). La afinidad `query·key` se pasa por `softmax` para obtener pesos que **suman 1**, y la salida es el promedio ponderado de los `value`. En una fórmula: `softmax(QK^T / √d) · V`. El `√d` evita que, con dimensiones grandes, el softmax se sature y mire a un solo token.

## 1. Scaled dot-product attention a mano en numpy

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
rng = np.random.default_rng(42)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def atencion(Q, K, V):
    d = Q.shape[-1]
    pesos = softmax(Q @ K.T / np.sqrt(d))   # softmax(QK^T / sqrt(d))
    return pesos @ V, pesos

seq_q, seq_k, d = 3, 4, 8
Q = rng.standard_normal((seq_q, d))
K = rng.standard_normal((seq_k, d))
V = rng.standard_normal((seq_k, d))
salida, pesos = atencion(Q, K, V)
print("salida:", salida.shape, "| pesos:", pesos.shape)

## 2. Por qué escalar por `√d`

In [ ]:
print("cada fila de pesos suma 1:", np.allclose(pesos.sum(axis=1), 1.0))
print("pesos[0]:", np.round(pesos[0], 3))

# Sin escalar, QK^T crece con d y el softmax se satura (una entropía menor).
sin_escala = softmax(Q @ K.T)
ent = lambda p: float(-(p * np.log(p + 1e-9)).sum(1).mean())
print("entropía con escala:", round(ent(pesos), 3),
      "| sin escala:", round(ent(sin_escala), 3))

## 3. `Attention` (Luong) y `AdditiveAttention` (Bahdanau) en Keras

In [ ]:
consulta = keras.Input(shape=(seq_q, d))
valores = keras.Input(shape=(seq_k, d))
attn = layers.Attention()([consulta, valores])              # dot-product (Luong)
add_attn = layers.AdditiveAttention()([consulta, valores])  # additive (Bahdanau)
print("Attention:", attn.shape, "| AdditiveAttention:", add_attn.shape)

## 4. `MultiHeadAttention` como self-attention

In [ ]:
x = keras.Input(shape=(seq_q, d))
mha = layers.MultiHeadAttention(num_heads=2, key_dim=4)
self_attn = mha(x, x)          # self-attention: query = value = x
print("self-attention:", self_attn.shape)
# Cada head aprende un patrón distinto en un subespacio de d.

## 5. Máscara causal (generación autoregresiva)

In [ ]:
causal = mha(x, x, use_causal_mask=True)   # cada posición solo mira <= t
print("self-attention causal:", causal.shape)
mask = np.tril(np.ones((seq_q, seq_q)))    # triangular inferior a mano
print("máscara causal:\n", mask.astype(int))

## 6. Cross-attention: el decoder mira al encoder + pesos

In [ ]:
q_dec = keras.Input(shape=(seq_q, d))
kv_enc = keras.Input(shape=(seq_k, d))
salida_cross, pesos_cross = layers.MultiHeadAttention(num_heads=2, key_dim=4)(
    q_dec, kv_enc, return_attention_scores=True)   # Q del decoder, K/V del encoder
print("cross-attention:", salida_cross.shape, "| pesos:", pesos_cross.shape)

## Ejercicios

1. **Attention a mano**: con Q, K, V random verificá shapes y que los pesos suman 1.
2. **Efecto del escalado**: compará la entropía de los pesos con y sin `/√d`.
3. **`MultiHeadAttention`**: aplicá `mha(x, x)` (self) y `mha(q, kv)` (cross) y compará shapes.
4. **Máscara causal**: usá `use_causal_mask=True` y verificá que no se mira el futuro.

## Conclusiones

- La atención pondera valores según la afinidad **query-key**: `softmax(QK^T/√d)·V`.
- El **escalado por √d** evita que el softmax se sature cuando `d` es grande.
- **Self-attention** (`Q=K=V`) es la base del Transformer; **cross-attention** conecta decoder→encoder.
- **Multi-head** corre varias atenciones en subespacios distintos y concatena.
- La máscara **causal** impide ver el futuro; los pesos son interpretables como alineamiento.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README. Como TensorFlow/PyTorch no están instalados en este entorno, las celdas de deep learning se validan por API (se ejecutan en Colab con GPU); las de **NumPy puro** son autónomas y traen `assert` para verificarse aquí mismo.

### Ejercicio 1 — Scaled dot-product attention a mano (NumPy, verificable)

`softmax(QK^T/√d)·V`. Se verifica el shape de la salida y que **cada fila de pesos suma 1**.

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def atencion(Q, K, V):
    d = Q.shape[-1]
    pesos = softmax(Q @ K.T / np.sqrt(d))
    return pesos @ V, pesos

rng = np.random.default_rng(42)
seq_q, seq_k, d = 3, 4, 8
Q, K, V = rng.standard_normal((seq_q, d)), rng.standard_normal((seq_k, d)), rng.standard_normal((seq_k, d))
salida, pesos = atencion(Q, K, V)
print("salida:", salida.shape, "| pesos:", pesos.shape)
assert salida.shape == (seq_q, d)
assert np.allclose(pesos.sum(axis=1), 1.0)        # cada query reparte 1.0 de atención

### Ejercicio 2 — Efecto del escalado por `√d` (NumPy, verificable)

Sin `/√d`, con `d` grande el producto crece y el softmax se **satura** (menor entropía → mira casi un solo token). Se compara la entropía media.

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

rng = np.random.default_rng(0)
d = 64
Q, K = rng.standard_normal((5, d)), rng.standard_normal((6, d))
con_escala = softmax(Q @ K.T / np.sqrt(d))
sin_escala = softmax(Q @ K.T)
entropia = lambda p: float(-(p * np.log(p + 1e-9)).sum(1).mean())
print("entropía con escala:", round(entropia(con_escala), 3),
      "| sin escala:", round(entropia(sin_escala), 3))
assert entropia(con_escala) > entropia(sin_escala)   # sin escalar el softmax se satura

### Ejercicio 3 — `MultiHeadAttention`: self vs cross

`mha(x, x)` es self-attention (Q=K=V); `mha(q, kv)` es cross-attention (decoder→encoder). La salida hereda la longitud de la **query**.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

seq_q, seq_k, d = 3, 5, 8
mha = layers.MultiHeadAttention(num_heads=2, key_dim=4)
x = keras.Input(shape=(seq_q, d))
q = keras.Input(shape=(seq_q, d)); kv = keras.Input(shape=(seq_k, d))
print("self  mha(x, x):", mha(x, x).shape)      # (None, seq_q, d)
print("cross mha(q,kv):", mha(q, kv).shape)     # (None, seq_q, d) — longitud de la query

### Ejercicio 4 — Máscara causal (no mirar el futuro)

`use_causal_mask=True` impide que la posición `t` atienda a `>t`. La máscara es una triangular inferior de 1s.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

seq_q, d = 4, 8
mha = layers.MultiHeadAttention(num_heads=2, key_dim=4)
x = keras.Input(shape=(seq_q, d))
causal = mha(x, x, use_causal_mask=True)
print("self-attention causal:", causal.shape)
mask = np.tril(np.ones((seq_q, seq_q), dtype=int))   # cada fila t ve columnas <= t
print("máscara causal:\n", mask)
assert (np.triu(mask, k=1) == 0).all()               # nada por encima de la diagonal